# 03 — Interaction Loop & Simulated User

**input:** `data/movies.pkl` and `data/embeddings.npy` (from notebook 01)

**Does one thing:**
1. Query retrieval — retrieves top-N nearest items from the full dataset
2. Interaction loop — clusters until reaching the stopping condition
3. Simulated user — always selects the cluster containing the target
4. Evaluation over N target items

**output:** `data/results_ours.pkl`

In [1]:
import numpy as np
import pandas as pd
import pickle, sys
from pathlib import Path
from sklearn.cluster import KMeans
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

sys.path.append(str(Path('../')))
from query_templates import get_query_for_movie

DATA_DIR    = Path('../data')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

movies     = pd.read_pickle(DATA_DIR / 'movies.pkl')
embeddings = np.load(DATA_DIR / 'embeddings.npy')

# load SBERT for encoding query text
sbert = SentenceTransformer('all-MiniLM-L6-v2')

TOP_N      = 500
STOP_SIZE  = 5
MAX_TURNS  = 6
N_EVAL     = 200

np.random.seed(42)
print('imports OK')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

imports OK


### Part 1 — helper functions from notebook 02 (copied)

In [2]:
import sys
sys.path.insert(0, '../src')

from core import (
    entropy,
    information_gain,
    adaptive_k,
    cluster_candidates,
    simulated_user
)

### Part 2 — Query Retrieval

In [3]:
def retrieve_candidates(query_embedding, all_embeddings, top_n):
    scores = all_embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_n]
    return top_indices.tolist()


def make_query_embedding(target_idx):
    """
    Builds a vague query from movie genres and encodes it.
    e.g.:  Animation|Children's|Comedy  →  'I want a funny animated movie for the family'
    """
    genres     = movies.iloc[target_idx]['genres']
    query_text = get_query_for_movie(genres)
    query_emb  = sbert.encode(query_text, normalize_embeddings=True)
    return query_emb, query_text


# ── quick test ──────────────────────────────────────────────────
target = 0  # Toy Story
q_emb, q_text = make_query_embedding(target)
candidates = retrieve_candidates(q_emb, embeddings, top_n=TOP_N)

print(f'Target : {movies.iloc[target]["title_clean"]}')
print(f'Query  : {q_text}')
print(f'Top 5 retrieved:')
for idx in candidates[:5]:
    print(f'  {movies.iloc[idx]["title_clean"]:40s}  {movies.iloc[idx]["genres_clean"]}')


Target : Toy Story
Query  : I want something fun for the whole family
Top 5 retrieved:
  Adventures in Babysitting                 Adventure Comedy
  Family Thing, A                           Comedy Drama
  My Family                                 Drama
  Fun and Fancy Free                        Animation Children's Musical
  Toys                                      Action Comedy Fantasy


### Part 3 — Simulated User

In [4]:
def simulated_user(partition: list[list[int]], target_idx: int) -> int:
    """
    Oracle user: always selects the cluster containing the target.
    If target is not in any cluster (should never happen), falls back to cluster 0.
    """
    for i, group in enumerate(partition):
        if target_idx in group:
            return i
    return 0

print('simulated_user defined')

simulated_user defined


### Part 4 — Full Interaction Loop

In [5]:
def run_interaction(target_idx, all_embeddings,
                    top_n=TOP_N, stop_size=STOP_SIZE,
                    max_turns=MAX_TURNS, verbose=False):

    # Step 1: query built from movie genre (intentionally vague)
    query_emb, query_text = make_query_embedding(target_idx)

    # Step 2: retrieval — top-N items nearest to the query
    C = retrieve_candidates(query_emb, all_embeddings, top_n=top_n)

    # if target is not in C, force-add it
    if target_idx not in C:
        C.append(target_idx)

    entropy_trace = [entropy(C)]
    ig_trace      = []
    turn          = 0

    while len(C) > stop_size and turn < max_turns:
        k = adaptive_k(len(C))
        partition, _ = cluster_candidates(C, all_embeddings, k=k)

        ig = information_gain(C, partition)
        ig_trace.append(ig)

        if verbose:
            print(f'  Turn {turn+1}: |C|={len(C)}, k={len(partition)}, IG={ig:.3f}')

        chosen = simulated_user(partition, target_idx)
        C = partition[chosen]
        turn += 1
        entropy_trace.append(entropy(C))

    return {
        'turns':         turn,
        'final_size':    len(C),
        'success':       target_idx in C,
        'entropy_trace': entropy_trace,
        'ig_trace':      ig_trace,
        'query_text':    query_text,
    }


# ── test on a single movie ──────────────────────────────────
target = 42
r = run_interaction(target, embeddings, verbose=True)
print()
print(f'Movie  : {movies.iloc[target]["title_clean"]}')
print(f'Query  : {r["query_text"]}')
print(f'Turns  : {r["turns"]}')
print(f'Final  : {r["final_size"]} items')
print(f'Success: {r["success"]}')


  Turn 1: |C|=501, k=5, IG=2.258
  Turn 2: |C|=116, k=5, IG=1.998
  Turn 3: |C|=42, k=4, IG=1.896
  Turn 4: |C|=15, k=3, IG=1.566

Movie  : Restoration
Query  : I want something that makes me feel something
Turns  : 4
Final  : 5 items
Success: True


In [6]:
# ── Load ratings as ground truth ─────────────────────────
ratings = pd.read_csv(
    DATA_DIR / 'ratings.dat',
    sep='::',
    engine='python',
    names=['user_id', 'movie_id', 'rating', 'timestamp'],
    encoding='latin-1'
)

# only ratings >= 4 — these are the true targets
high_ratings = ratings[ratings['rating'] >= 4].copy()

# convert movie_id to positional index
movie_id_to_idx = {mid: idx for idx, mid in enumerate(movies['movie_id'])}
high_ratings['movie_idx'] = high_ratings['movie_id'].map(movie_id_to_idx)
high_ratings = high_ratings.dropna(subset=['movie_idx'])
high_ratings['movie_idx'] = high_ratings['movie_idx'].astype(int)

print(f'Total high ratings (>=4): {len(high_ratings)}')
print(f'Unique movies as targets: {high_ratings["movie_idx"].nunique()}')

# sample N_EVAL targets
sample = high_ratings.sample(n=N_EVAL, random_state=42)
target_items = sample['movie_idx'].values

print(f'Sample size: {N_EVAL}')
print('\nExample targets:')
for idx in target_items[:3]:
    m = movies.iloc[idx]
    print(f'  {m["title_clean"]:40s}  {m["genres_clean"]}')


Total high ratings (>=4): 575281
Unique movies as targets: 3533
Sample size: 200

Example targets:
  Men in Black                              Action Adventure Comedy Sci-Fi
  Gremlins                                  Comedy Horror
  Nineteen Eighty-Four                      Drama Sci-Fi


### Part 6 — Run Evaluation

In [7]:
# ── Run evaluation ───────────────────────────────────────
results = []
for t in tqdm(target_items, desc='Our method'):
    r = run_interaction(int(t), embeddings)
    r['target_idx'] = int(t)
    results.append(r)

df = pd.DataFrame(results)

print('\n── Results ─────────────────────────────')
print(f'Avg turns     : {df["turns"].mean():.2f}')
print(f'Avg final size: {df["final_size"].mean():.2f}')
print(f'Success rate  : {df["success"].mean():.3f}')
print(f'  (success = target survived to final candidate set)')

with open(DATA_DIR / 'results_ours.pkl', 'wb') as f:
    pickle.dump({'df': df, 'target_items': target_items}, f)

print('\nSaved: data/results_ours.pkl')
print('✅ notebook 03 complete')


Our method: 100%|██████████| 200/200 [00:10<00:00, 18.23it/s]


── Results ─────────────────────────────
Avg turns     : 4.44
Avg final size: 3.56
Success rate  : 1.000
  (success = target survived to final candidate set)

Saved: data/results_ours.pkl
✅ notebook 03 complete
